# TP corrigé — PCA et évaluation d’un modèle avant / après PCA

**Module : Introduction à l’IA**  
**Objectif du TP :** comprendre l’effet de la PCA sur un modèle de classification.

Dans ce notebook, on compare deux approches :

1. **Modèle avant PCA** : le modèle utilise toutes les variables après préparation des données.
2. **Modèle après PCA** : les variables sont d’abord transformées en composantes principales, puis le modèle est entraîné.

> Point méthodologique essentiel : on fait toujours le **train-test split avant la PCA**.  
> La PCA doit être apprise uniquement sur les données d’entraînement pour éviter la fuite d’information (*data leakage*).

## 1. Importation des bibliothèques

On importe les bibliothèques nécessaires pour :
- lire les données ;
- préparer les variables numériques et catégorielles ;
- entraîner les modèles ;
- appliquer la PCA ;
- évaluer les résultats.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

## 2. Chargement des données

Le TP utilise le dataset **Loan eligibility**.  
Le code ci-dessous essaie d’abord de lire un fichier local `train.csv`. S’il n’existe pas, il lit directement le fichier depuis GitHub.

In [ ]:
url = "https://raw.githubusercontent.com/profsarang/ThinkingDataScience/main/data/train.csv"

try:
    df = pd.read_csv("train.csv")
    print("Données chargées depuis le fichier local train.csv")
except FileNotFoundError:
    df = pd.read_csv(url)
    print("Données chargées depuis GitHub")

df.head()

## 3. Exploration rapide des données

On vérifie la taille du dataset, les types des colonnes et les valeurs manquantes.

In [ ]:
print("Dimensions du dataset :", df.shape)
df.info()

In [ ]:
missing_table = pd.DataFrame({
    "Valeurs manquantes": df.isnull().sum(),
    "Pourcentage (%)": (df.isnull().mean() * 100).round(2)
})
missing_table

## 4. Séparation entre variables explicatives et variable cible

La variable cible est `Loan_Status`.  
On enlève aussi `Loan_ID` car c’est un identifiant, pas une variable utile pour la prédiction.

In [ ]:
target_col = "Loan_Status"

X = df.drop(columns=[target_col])
y = df[target_col]

# Supprimer l'identifiant s'il existe
if "Loan_ID" in X.columns:
    X = X.drop(columns=["Loan_ID"])

print("Variables explicatives X :", X.shape)
print("Variable cible y :", y.shape)
X.head()

## 5. Train-test split avant toute transformation

On sépare les données en deux parties :

- **Train** : utilisé pour apprendre le modèle ;
- **Test** : utilisé uniquement pour évaluer le modèle.

Cette étape doit être faite **avant** la standardisation et avant la PCA.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

## 6. Préparation automatique des données

Les variables numériques et catégorielles ne sont pas traitées de la même façon.

Pour les variables numériques :
- remplacement des valeurs manquantes par la médiane.

Pour les variables catégorielles :
- remplacement des valeurs manquantes par la valeur la plus fréquente ;
- encodage avec One-Hot Encoding.

Ensuite, on applique `StandardScaler`, car la PCA est sensible à l’échelle des variables.

In [ ]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Variables numériques :", numeric_features)
print("Variables catégorielles :", categorical_features)

In [ ]:
# Compatibilité avec différentes versions de scikit-learn
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", encoder)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 7. Fonction d’évaluation

Cette fonction entraîne un modèle, prédit sur le test set, puis calcule :

- Accuracy ;
- Precision macro ;
- Recall macro ;
- F1-score macro.

On affiche aussi le rapport de classification.

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    """Entraîne un modèle et retourne ses métriques d'évaluation."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results = {
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0)
    }

    print("=" * 70)
    print(name)
    print("=" * 70)
    print(classification_report(y_test, y_pred, zero_division=0))

    return results, y_pred

# Partie A — Modèle avant PCA

Dans cette première expérience, le modèle utilise toutes les variables préparées.

Pipeline utilisé :

```text
Prétraitement → Standardisation → Logistic Regression
```

C’est notre modèle de référence.

In [ ]:
model_before_pca = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

results_before, y_pred_before = evaluate_model(
    "Avant PCA - Logistic Regression",
    model_before_pca,
    X_train,
    X_test,
    y_train,
    y_test
)

In [ ]:
ConfusionMatrixDisplay.from_estimator(model_before_pca, X_test, y_test)
plt.title("Matrice de confusion - Avant PCA")
plt.show()

# Partie B — Modèle après PCA

Dans cette deuxième expérience, on ajoute la PCA avant l’entraînement du modèle.

Pipeline utilisé :

```text
Prétraitement → Standardisation → PCA → Logistic Regression
```

Ici, `n_components=0.95` signifie que la PCA garde le nombre de composantes nécessaires pour conserver environ **95 % de la variance**.

In [ ]:
model_after_pca_95 = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

results_after_95, y_pred_after_95 = evaluate_model(
    "Après PCA - 95% variance",
    model_after_pca_95,
    X_train,
    X_test,
    y_train,
    y_test
)

In [ ]:
# Nombre de composantes réellement conservées
pca_95 = model_after_pca_95.named_steps["pca"]

print("Nombre de composantes conservées :", pca_95.n_components_)
print("Variance totale conservée :", round(pca_95.explained_variance_ratio_.sum() * 100, 2), "%")

In [ ]:
ConfusionMatrixDisplay.from_estimator(model_after_pca_95, X_test, y_test)
plt.title("Matrice de confusion - Après PCA 95% variance")
plt.show()

# Partie C — PCA avec seulement 2 composantes

Cette partie sert surtout à voir l’effet de la PCA pour la visualisation.

Avec 2 composantes, on peut représenter le dataset dans un graphique 2D.  
Mais attention : garder seulement 2 composantes peut réduire fortement l’information disponible pour le modèle.

In [ ]:
model_after_pca_2 = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2, random_state=42)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

results_after_2, y_pred_after_2 = evaluate_model(
    "Après PCA - 2 composantes",
    model_after_pca_2,
    X_train,
    X_test,
    y_train,
    y_test
)

In [ ]:
ConfusionMatrixDisplay.from_estimator(model_after_pca_2, X_test, y_test)
plt.title("Matrice de confusion - Après PCA 2 composantes")
plt.show()

## Visualisation en 2D après PCA

La PCA transforme les variables originales en deux nouveaux axes :

- **PC1** : première composante principale ;
- **PC2** : deuxième composante principale.

On affiche ici les données de test projetées sur ces deux composantes.

In [ ]:
# Pipeline sans classifieur : uniquement préparation + PCA 2D
pca_visualization = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2, random_state=42))
])

# Fit uniquement sur le train, puis transformation du test
X_train_pca_2d = pca_visualization.fit_transform(X_train)
X_test_pca_2d = pca_visualization.transform(X_test)

plt.figure(figsize=(8, 6))
for label in np.unique(y_test):
    mask = (y_test == label).values if hasattr(y_test, "values") else (y_test == label)
    plt.scatter(X_test_pca_2d[mask, 0], X_test_pca_2d[mask, 1], label=str(label), alpha=0.75)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Projection des données de test avec PCA en 2D")
plt.legend(title="Loan_Status")
plt.grid(True)
plt.show()

# Partie D — Comparaison des résultats

On compare maintenant les performances :

- modèle sans PCA ;
- modèle avec PCA conservant 95 % de la variance ;
- modèle avec PCA en 2 composantes.

In [ ]:
comparison = pd.DataFrame([
    results_before,
    results_after_95,
    results_after_2
]).set_index("Modèle")

comparison.round(3)

In [ ]:
comparison[["Accuracy", "Precision_macro", "Recall_macro", "F1_macro"]].plot(
    kind="bar",
    figsize=(11, 5)
)
plt.title("Comparaison des modèles avant et après PCA")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y")
plt.show()

# Partie E — Effet du nombre de composantes PCA

On teste plusieurs valeurs de `n_components` pour voir comment le nombre de composantes influence la performance.

Cette partie montre que :
- trop peu de composantes peuvent supprimer de l’information ;
- trop de composantes peuvent réduire l’intérêt de la PCA ;
- il faut chercher un équilibre entre simplification et performance.

In [ ]:
# On calcule d'abord le nombre de colonnes après prétraitement, en utilisant seulement le train
X_train_processed = preprocessor.fit_transform(X_train)
n_features_after_preprocessing = X_train_processed.shape[1]

max_components = min(15, n_features_after_preprocessing)
print("Nombre de variables après prétraitement :", n_features_after_preprocessing)
print("Nombre maximum de composantes testées :", max_components)

In [ ]:
component_results = []

for k in range(1, max_components + 1):
    model_k = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=k, random_state=42)),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42))
    ])

    model_k.fit(X_train, y_train)
    y_pred_k = model_k.predict(X_test)

    component_results.append({
        "n_components": k,
        "Accuracy": accuracy_score(y_test, y_pred_k),
        "F1_macro": f1_score(y_test, y_pred_k, average="macro", zero_division=0),
        "Variance_expliquée": model_k.named_steps["pca"].explained_variance_ratio_.sum()
    })

component_results = pd.DataFrame(component_results)
component_results.round(3)

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(component_results["n_components"], component_results["Accuracy"], marker="o", label="Accuracy")
plt.plot(component_results["n_components"], component_results["F1_macro"], marker="s", label="F1 macro")
plt.xlabel("Nombre de composantes PCA")
plt.ylabel("Score")
plt.title("Performance selon le nombre de composantes PCA")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(component_results["n_components"], component_results["Variance_expliquée"], marker="o")
plt.xlabel("Nombre de composantes PCA")
plt.ylabel("Variance expliquée cumulée")
plt.title("Variance expliquée selon le nombre de composantes")
plt.ylim(0, 1.05)
plt.grid(True)
plt.show()

# Conclusion pédagogique

À la fin de ce TP, il faut retenir que :

1. La PCA s’applique **après le train-test split**.
2. La PCA doit être apprise sur `X_train` uniquement.
3. On applique ensuite la même transformation à `X_test`.
4. La PCA peut simplifier les données et réduire le bruit.
5. Mais elle peut aussi faire baisser la performance si elle supprime trop d’information.

Phrase à retenir :

> **La PCA réduit la dimension des données, mais son effet doit toujours être vérifié par une évaluation du modèle avant et après PCA.**

## Travail demandé aux étudiants

1. Expliquer pourquoi le split doit être fait avant la PCA.
2. Comparer les résultats du modèle avant et après PCA.
3. Dire si la PCA améliore ou diminue la performance dans ce cas.
4. Identifier le meilleur nombre de composantes selon le graphique.
5. Expliquer la différence entre la PCA utilisée pour la visualisation et la PCA utilisée pour améliorer un modèle.